# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedant08mehta/Flyrank-assignment1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/vedant08mehta/Flyrank-assignment1.git
%cd /content/Flyrank-assignment1

Cloning into 'Flyrank-assignment1'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 156 (delta 63), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 1.89 MiB | 11.69 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/Flyrank-assignment1


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use Random Forest classification because the goal is to identify pages that are likely to be declining. Random Forest can capture non-linear relationships between search-performance features and the decline label while still providing feature importance for interpretation. It is also suitable for comparing a learned model against the hand-written baseline from Week 4.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(f"Rows: {len(df):,}")
print(f"Declining pages: {df['is_declining_label'].sum():,}")
print(
    f"Declining rate: "
    f"{df['is_declining_label'].mean():.3f}"
)

Rows: 30,000
Declining pages: 16,262
Declining rate: 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-grouped split so that pages belonging to the same client do not appear in both training and test sets. This gives a more realistic estimate of how the model could perform on unseen clients and is consistent with the client-holdout approach used by the reference pipeline.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update"
]

model_data = df.dropna(
    subset=feature_cols + ["is_declining_label", "client_id"]
).copy()

X = model_data[feature_cols]
y = model_data["is_declining_label"]
groups = model_data["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print(f"Train rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

print(
    f"Train clients: "
    f"{groups_train.nunique():,}"
)

print(
    f"Test clients: "
    f"{groups_test.nunique():,}"
)

print(
    "Client overlap:",
    len(
        set(groups_train)
        & set(groups_test)
    )
)

Train rows: 22,885
Test rows: 7,115
Train clients: 24
Test clients: 8
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will train the Random Forest on the training clients only and evaluate it on the unseen test clients. I will compare its Precision@50 with the Week-4 hand-built baseline using the same test set and the same target definition. This makes the comparison more meaningful than comparing models on different samples.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import precision_score

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(
    X_train,
    y_train
)

test_proba = model.predict_proba(
    X_test
)[:, 1]

top50_idx = np.argsort(
    test_proba
)[::-1][:50]

model_precision_50 = y_test.iloc[
    top50_idx
].mean()

print(
    f"Model Precision@50: "
    f"{model_precision_50:.3f}"
)

print(
    f"Test-set base rate: "
    f"{y_test.mean():.3f}"
)

Model Precision@50: 0.680
Test-set base rate: 0.517


In [6]:
baseline_score = (
    (1 - X_test["ctr"].rank(pct=True))
    + X_test["avg_position"].rank(pct=True)
)

baseline_top50 = baseline_score.nlargest(50).index

baseline_precision_50 = y_test.loc[
    baseline_top50
].mean()

print(
    f"Baseline Precision@50: "
    f"{baseline_precision_50:.3f}"
)

print(
    f"Model Precision@50: "
    f"{model_precision_50:.3f}"
)

print(
    f"Lift: "
    f"{model_precision_50 - baseline_precision_50:+.3f}"
)

Baseline Precision@50: 0.380
Model Precision@50: 0.680
Lift: +0.300


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model performs better than the hand-written baseline on the held-out test clients, but it still makes both false-positive and false-negative predictions. This suggests that the model captures useful patterns that the simple rule misses, while some pages remain difficult to classify. The feature importance results also help show which available signals the model relies on most. These results should be treated as decision-support rather than proof of causation.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix

test_results = X_test.copy()

test_results["actual"] = y_test
test_results["prediction_score"] = test_proba
test_results["prediction"] = (
    test_proba >= 0.5
).astype(int)

test_results["error_type"] = np.select(
    [
        (
            (test_results["actual"] == 0)
            & (test_results["prediction"] == 1)
        ),
        (
            (test_results["actual"] == 1)
            & (test_results["prediction"] == 0)
        )
    ],
    [
        "False positive",
        "False negative"
    ],
    default="Correct"
)

print("=== Error counts ===")

print(
    test_results["error_type"].value_counts()
)

print("\n=== Feature importance ===")

importance = (
    pd.Series(
        model.feature_importances_,
        index=feature_cols
    )
    .sort_values(ascending=False)
)

display(
    importance.to_frame("importance")
)

print("\n=== Highest-confidence false positives ===")

display(
    test_results[
        test_results["error_type"] == "False positive"
    ]
    .sort_values(
        "prediction_score",
        ascending=False
    )
    .head(10)
)

=== Error counts ===
error_type
Correct           4051
False positive    1856
False negative    1208
Name: count, dtype: int64

=== Feature importance ===


,importance
impressions_90d,0.266065
avg_position,0.252311
content_age_days,0.196112
ctr,0.100590
clicks_90d,0.076846
engagement_rate,0.062520
days_since_last_update,0.045556



=== Highest-confidence false positives ===


,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,content_age_days,days_since_last_update,actual,prediction_score,prediction,error_type
4878,165,0,0.0,13.0,0.0,180,20,0,0.995,1,False positive
1439,106,0,0.0,18.2,0.0,223,102,0,0.985,1,False positive
21291,179,0,0.0,6.9,0.0,144,20,0,0.985,1,False positive
7224,740,0,0.0,12.4,0.0,141,20,0,0.980,1,False positive
1634,428,0,0.0,8.2,0.0,144,20,0,0.980,1,False positive
22812,701,0,0.0,12.8,0.0,95,20,0,0.975,1,False positive
7537,222,0,0.0,12.4,0.0,182,20,0,0.975,1,False positive
13561,134,0,0.0,9.3,0.0,275,104,0,0.975,1,False positive
18523,905,0,0.0,5.6,0.0,151,20,0,0.975,1,False positive
10351,230,0,0.0,7.7,0.0,281,20,0,0.975,1,False positive


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.